# Thermal TFIM analysis

This notebook uses the shared scripts in `scripts/` and saves paper figures under `Paper plots Thermal/`.

In [16]:
from pathlib import Path
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import numpy as np

def find_repo_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "scripts" / "data_config.py").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root containing scripts/data_config.py")

REPO_ROOT = find_repo_root(Path.cwd())
NOTEBOOK_DIR = REPO_ROOT / "Paper Analysis" / "TFIM Thermal"
SCRIPTS_DIR = REPO_ROOT / "scripts"
PLOTS_DIR = NOTEBOOK_DIR / "Paper plots Thermal"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from plot_gram_matrices import (
    _plot_thermal_gram_on_axis,
    _set_normalized_cbar_ticks,
    _thermal_cbar_label,
    compute_thermal_gram_matrix,
    plot_thermal_gram_matrices,
)
from plot_ipr_curves import set_ipr_plot_style
from run_thermal_kernel_svm import run_thermal_kernel_svm_beta_sweep

def save_png(fig, stem):
    path = PLOTS_DIR / f"{stem}.png"
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return path


## Thermal Gram matrices at L=100

In [ ]:
def plot_thermal_gram_matrix_beta_panel():
    set_ipr_plot_style()
    betas = [0.1, 0.2, 0.3, 0.4, 2.0]
    rows = [("hs", "HS"), ("ipr", "IPR")]
    norm = Normalize(vmin=0.0, vmax=1.0)

    fig, axes = plt.subplots(
        len(rows),
        len(betas),
        figsize=(13.0, 5.2),
        sharex=True,
        sharey=True,
        constrained_layout=False,
    )

    last_im = None
    for row, (kind, row_label) in enumerate(rows):
        for col, beta in enumerate(betas):
            ax = axes[row, col]
            gram = compute_thermal_gram_matrix(
                kind=kind,
                L=100,
                beta=beta,
                source="processed",
                normalize=True,
            )
            last_im = _plot_thermal_gram_on_axis(ax, gram, norm=norm)
            if row == 0:
                ax.set_title(rf"$\beta={beta:g}$", pad=5)
            if col == 0:
                ax.set_ylabel(f"{row_label}\n$h_b$")
            ax.set_xlabel(r"$h_a$")
            ax.tick_params(labelbottom=True)

    fig.tight_layout(rect=[0.0, 0.0, 0.92, 1.0])
    if last_im is not None:
        cax = fig.add_axes([0.935, 0.18, 0.014, 0.64])
        cbar = fig.colorbar(last_im, cax=cax)
        _set_normalized_cbar_ticks(cbar, normalized=True)
        cbar.set_label(r"$K_{\mathrm{norm}}$")
    return fig, axes


fig, axes = plot_thermal_gram_matrix_beta_panel()
save_png(fig, "thermal_L100_gram_matrices_beta_panel")

## Thermal Gram matrices across L

In [ ]:
THERMAL_GRAM_L_SWEEP_BETA = 0.2


def beta_filename_part(beta):
    return f"{float(beta):g}".replace(".", "p")


def plot_thermal_gram_l_sweep(kind, beta=THERMAL_GRAM_L_SWEEP_BETA):
    set_ipr_plot_style()
    L_values = list(range(10, 101, 10))
    norm = Normalize(vmin=0.0, vmax=1.0)

    fig, axes = plt.subplots(
        2,
        5,
        figsize=(13.0, 5.3),
        sharex=True,
        sharey=True,
        constrained_layout=False,
    )
    axes_flat = axes.ravel()

    last_im = None
    for idx, (ax, L) in enumerate(zip(axes_flat, L_values)):
        row = idx // 5
        col = idx % 5
        gram = compute_thermal_gram_matrix(
            kind=kind,
            L=L,
            beta=beta,
            source="raw",
            normalize=True,
        )
        last_im = _plot_thermal_gram_on_axis(ax, gram, norm=norm)
        ax.text(
            0.04,
            0.92,
            rf"$L={L}$",
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=11,
            bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.72, "pad": 1.5},
        )
        ax.set_xlabel(r"$h_a$")
        ax.tick_params(labelbottom=True)
        if col == 0:
            ax.set_ylabel(r"$h_b$")
        else:
            ax.set_ylabel("")

    fig.tight_layout(rect=[0.0, 0.0, 0.92, 0.94])
    if last_im is not None:
        cax = fig.add_axes([0.935, 0.18, 0.014, 0.64])
        cbar = fig.colorbar(last_im, cax=cax)
        _set_normalized_cbar_ticks(cbar, normalized=True)
        cbar.set_label(_thermal_cbar_label(kind, normalized=True))
    return fig, axes


fig, axes = plot_thermal_gram_l_sweep("ipr", beta=THERMAL_GRAM_L_SWEEP_BETA)
save_png(fig, f"thermal_ipr_gram_L_sweep_beta_{beta_filename_part(THERMAL_GRAM_L_SWEEP_BETA)}")

fig, axes = plot_thermal_gram_l_sweep("hs", beta=THERMAL_GRAM_L_SWEEP_BETA)
save_png(fig, f"thermal_hs_gram_L_sweep_beta_{beta_filename_part(THERMAL_GRAM_L_SWEEP_BETA)}")

## Thermal Gram matrices across beta

In [ ]:
THERMAL_GRAM_BETA_SWEEP_L = 20
THERMAL_GRAM_BETA_SWEEP_VALUES = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 1.0, 3.0, 6.0]

fig, axes, grams = plot_thermal_gram_matrices(
    kind="ipr",
    L=THERMAL_GRAM_BETA_SWEEP_L,
    betas=THERMAL_GRAM_BETA_SWEEP_VALUES,
    source="raw",
    normalize=True,
    ncols=3,
    panel_size=(2.8, 2.6),
    show_title=False,
)
save_png(fig, f"thermal_ipr_gram_beta_sweep_L{THERMAL_GRAM_BETA_SWEEP_L}")

fig, axes, grams = plot_thermal_gram_matrices(
    kind="hs",
    L=THERMAL_GRAM_BETA_SWEEP_L,
    betas=THERMAL_GRAM_BETA_SWEEP_VALUES,
    source="raw",
    normalize=True,
    ncols=3,
    panel_size=(2.8, 2.6),
    show_title=False,
)
save_png(fig, f"thermal_hs_gram_beta_sweep_L{THERMAL_GRAM_BETA_SWEEP_L}")

## Thermal SVM balanced accuracy, stacked by system size

In [ ]:
thermal_svm_results_by_L = {}
for L in [20, 50, 100]:
    fig, ax, results = run_thermal_kernel_svm_beta_sweep(
        L=L,
        kinds=("ipr", "hs"),
        source="processed",
        test_frac=0.10,
        local_delta=0.25,
        n_splits=100,
        rng_seed=12345,
        C=1.0,
        show_plot_title=False,
    )
    plt.close(fig)
    thermal_svm_results_by_L[L] = results

In [ ]:
def plot_stacked_thermal_svm(results_by_L):
    set_ipr_plot_style()
    L_values = [20, 50, 100]
    colors = {"ipr": "#1b9e77", "hs": "#d95f02"}
    markers = {"ipr": "s", "hs": "o"}

    fig, axes = plt.subplots(
        len(L_values),
        1,
        figsize=(7.4, 7.0),
        sharex=True,
        sharey=True,
        constrained_layout=True,
    )

    for ax, L in zip(axes, L_values):
        results = results_by_L[L]
        betas = np.asarray(results["betas"], dtype=float)
        for kind in ("ipr", "hs"):
            res = results["by_kind"][kind]
            color = colors[kind]
            marker = markers[kind]
            label_prefix = kind.upper()
            ax.errorbar(
                betas,
                res["global_bal_acc_mean"],
                yerr=res["global_bal_acc_std"],
                marker=marker,
                lw=1.8,
                capsize=2.5,
                ls="-",
                color=color,
                label=f"{label_prefix} global",
            )
            ax.errorbar(
                betas,
                res["local_bal_acc_mean"],
                yerr=res["local_bal_acc_std"],
                marker=marker,
                lw=1.8,
                capsize=2.5,
                ls="--",
                color=color,
                label=f"{label_prefix} local",
            )
        ax.text(0.03, 0.12, rf"$L={L}$", transform=ax.transAxes, ha="left", va="bottom", fontsize=13)
        ax.set_ylim(0.6, 1.03)
        ax.set_xlim(2.0, 0.0)
        ax.set_yticks(np.linspace(0.6, 1.0, 5))
        ax.tick_params(direction="in", which="both", top=True, right=True)
        ax.grid(which="major", alpha=0.18, lw=0.7)
        ax.grid(which="minor", alpha=0.07, lw=0.5)
        ax.minorticks_on()
        ax.legend(frameon=False, loc="lower left", bbox_to_anchor=(0.16, 0.02), ncol=2, fontsize=9, handlelength=2.4, columnspacing=1.0)

    axes[-1].set_xlabel(r"$\beta$")
    axes[1].set_ylabel("Balanced accuracy")
    axes[-1].set_xticks(np.arange(2.0, -1e-9, -0.2))
    return fig, axes

fig, axes = plot_stacked_thermal_svm(thermal_svm_results_by_L)
save_png(fig, "thermal_svm_stacked_L20_L50_L100")

## Thermal SVM results table

In [ ]:
def format_mean_pm(mean, std):
    return f"{float(mean):.4f} $\\pm$ {float(std):.4f}"


def thermal_svm_results_latex_table(results_by_L):
    lines = [
        r"\begin{tabular}{rrllllr}",
        r"\toprule",
        r"$L$ & $\beta$ & IPR global & IPR local & HS global & HS local & Splits \\",
        r"\midrule",
    ]

    for L in [20, 50, 100]:
        results = results_by_L[L]
        betas = np.asarray(results["betas"], dtype=float)
        ipr = results["by_kind"]["ipr"]
        hs = results["by_kind"]["hs"]
        for idx, beta in enumerate(betas):
            lines.append(
                f"{L} & {beta:g} & "
                f"{format_mean_pm(ipr['global_bal_acc_mean'][idx], ipr['global_bal_acc_std'][idx])} & "
                f"{format_mean_pm(ipr['local_bal_acc_mean'][idx], ipr['local_bal_acc_std'][idx])} & "
                f"{format_mean_pm(hs['global_bal_acc_mean'][idx], hs['global_bal_acc_std'][idx])} & "
                f"{format_mean_pm(hs['local_bal_acc_mean'][idx], hs['local_bal_acc_std'][idx])} & "
                f"{int(results['n_splits'])} \\\\"
            )
        if L != 100:
            lines.append(r"\midrule")

    lines.extend([r"\bottomrule", r"\end{tabular}"])
    return "\n".join(lines) + "\n"


thermal_svm_latex_path = PLOTS_DIR / "thermal_svm_results_L20_L50_L100_latex.txt"
thermal_svm_latex_path.write_text(thermal_svm_results_latex_table(thermal_svm_results_by_L), encoding="utf-8")
thermal_svm_latex_path

## Decision-functions
 
 This plot overlays the stored decision functions for every beta. Each curve is the split-averaged SVM decision function evaluated over the full $h$ grid.
 
 The $x$-axis is $\beta * (h - h_c)$. The width of the transition region should scale as $\delta h \approx \frac{1}{\beta}$. Therefore we expect the curves to collapse on each other if they properly reflect that critical scaling.

 Beta is encoded by opacity: smaller $\beta$ curves are fainter, while $\beta=2$ uses the full standard IPR green or HS orange.




In [ ]:
def plot_stacked_global_thermal_decision_functions(results_by_L):
    set_ipr_plot_style()
    L_values = [20, 50, 100]
    kinds = ["ipr", "hs"]
    colors = {"ipr": "#1b9e77", "hs": "#d95f02"}
    all_betas = np.asarray(results_by_L[L_values[0]]["betas"], dtype=float)
    titles = {"ipr": "IPR global", "hs": "HS global"}

    fig, axes = plt.subplots(
        len(L_values),
        len(kinds),
        figsize=(9.2, 7.0),
        sharex=True,
        sharey=True,
        constrained_layout=True,
    )

    for row, L in enumerate(L_values):
        results = results_by_L[L]
        betas = np.asarray(results["betas"], dtype=float)
        beta_min = float(np.min(betas))
        beta_max = float(np.max(betas))
        beta_span = beta_max - beta_min if beta_max > beta_min else 1.0

        for col, kind in enumerate(kinds):
            ax = axes[row, col]
            metadata = results["by_kind"][kind]["gram_metadata"]
            first_line = True
            for meta in sorted(metadata, key=lambda item: float(item["beta"]), reverse=True):
                beta = float(meta["beta"])
                decision = meta["decision_function"]
                x = beta * (np.asarray(decision["x"], dtype=float) - float(results["phase_boundary"]))
                y = np.asarray(decision["global_mean"], dtype=float)
                y_scale = np.nanmax(np.abs(y))
                if np.isfinite(y_scale) and y_scale > 0:
                    y = y / y_scale
                beta_weight = (beta - beta_min) / beta_span
                alpha = 0.12 + 0.83 * beta_weight
                ax.plot(
                    x,
                    y,
                    color=colors[kind],
                    lw=1.15,
                    alpha=alpha,
                    label=None,
                )
                first_line = False

            ax.axhline(0.0, color="0.35", lw=0.8, ls=":")
            ax.axvline(0.0, color="0.35", lw=0.8, ls=":")
            ax.text(0.03, 0.92, rf"$L={L}$", transform=ax.transAxes, ha="left", va="top", fontsize=13)
            ax.set_xlim(-2.0, 2.0)
            ax.set_ylim(-1.05, 1.05)
            ax.tick_params(direction="in", which="both", top=True, right=True)
            ax.grid(which="major", alpha=0.18, lw=0.7)
            ax.grid(which="minor", alpha=0.07, lw=0.5)
            ax.minorticks_on()

    for ax in axes[-1, :]:
        ax.set_xlabel(r"$\beta(h-h_c)$")
    axes[1, 0].set_ylabel("normalized decision function")
    axes[0, 0].set_title("IPR global", pad=6)
    axes[0, 1].set_title("HS global", pad=6)
    return fig, axes

fig, axes = plot_stacked_global_thermal_decision_functions(thermal_svm_results_by_L)
save_png(fig, "thermal_global_decision_functions_stacked_L20_L50_L100")

## L=100 decision-function comparison

In [ ]:
def plot_L100_global_thermal_decision_function_comparison(results_by_L):
    set_ipr_plot_style()
    L = 100
    kinds = ["ipr", "hs"]
    colors = {"ipr": "#1b9e77", "hs": "#d95f02"}

    results = results_by_L[L]
    betas = np.asarray(results["betas"], dtype=float)
    beta_min = float(np.min(betas))
    beta_max = float(np.max(betas))
    beta_span = beta_max - beta_min if beta_max > beta_min else 1.0

    fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.4), sharex=True, sharey=True, constrained_layout=True)
    for ax, kind in zip(axes, kinds):
        metadata = results["by_kind"][kind]["gram_metadata"]
        for meta in sorted(metadata, key=lambda item: float(item["beta"]), reverse=True):
            beta = float(meta["beta"])
            decision = meta["decision_function"]
            x = beta * (np.asarray(decision["x"], dtype=float) - float(results["phase_boundary"]))
            y = np.asarray(decision["global_mean"], dtype=float)
            y_scale = np.nanmax(np.abs(y))
            if np.isfinite(y_scale) and y_scale > 0:
                y = y / y_scale
            beta_weight = (beta - beta_min) / beta_span
            alpha = 0.12 + 0.83 * beta_weight
            ax.plot(x, y, color=colors[kind], lw=1.15, alpha=alpha)

        ax.axhline(0.0, color="0.35", lw=0.8, ls=":")
        ax.axvline(0.0, color="0.35", lw=0.8, ls=":")
        ax.text(0.03, 0.92, rf"$L={L}$", transform=ax.transAxes, ha="left", va="top", fontsize=13)
        ax.set_xlim(-2.0, 2.0)
        ax.set_ylim(-1.05, 1.05)
        ax.tick_params(direction="in", which="both", top=True, right=True)
        ax.grid(which="major", alpha=0.18, lw=0.7)
        ax.grid(which="minor", alpha=0.07, lw=0.5)
        ax.minorticks_on()

    for ax in axes:
        ax.set_xlabel(r"$\beta(h-h_c)$")
    axes[0].set_ylabel("normalized decision function")
    axes[0].set_title("IPR global", pad=6)
    axes[1].set_title("HS global", pad=6)
    return fig, axes


fig, axes = plot_L100_global_thermal_decision_function_comparison(thermal_svm_results_by_L)
save_png(fig, "thermal_global_decision_functions_L100_ipr_hs_comparison")

## Thermal global SVM sharpness

We want to analyse the "suddenness" of change in the decision function, where a steeper decline means a better capturing of the phase transition. Here $h^*(\beta)$ is the learned zero crossing of the global decision function closest to $h_c=1$, where the curve crosses from positive to negative. The plot shows the sharpness $S(\beta)=|\partial_h f(h,\beta)|_{h=h^*(\beta)}$ versus $\beta$ for IPR and HS, separately for $L=20,50,100$.

In [ ]:
def find_decision_boundary(h_grid, f_grid, h_c=1.0):
    h_grid = np.asarray(h_grid, dtype=float)
    f_grid = np.asarray(f_grid, dtype=float)
    finite = np.isfinite(h_grid) & np.isfinite(f_grid)
    h_grid = h_grid[finite]
    f_grid = f_grid[finite]
    if h_grid.size < 2:
        return np.nan

    order = np.argsort(h_grid)
    h_grid = h_grid[order]
    f_grid = f_grid[order]

    crossings = np.where((f_grid[:-1] < 0.0) & (f_grid[1:] > 0.0))[0]
    if crossings.size == 0:
        return np.nan

    h_stars = []
    for i in crossings:
        h0, h1 = h_grid[i], h_grid[i + 1]
        f0, f1 = f_grid[i], f_grid[i + 1]
        h_stars.append(float(h0 - f0 * (h1 - h0) / (f1 - f0)))

    h_stars = np.asarray(h_stars, dtype=float)
    return float(h_stars[np.argmin(np.abs(h_stars - float(h_c)))])


def decision_derivative_at(h_grid, f_grid, h_eval):
    h_grid = np.asarray(h_grid, dtype=float)
    f_grid = np.asarray(f_grid, dtype=float)
    finite = np.isfinite(h_grid) & np.isfinite(f_grid)
    h_grid = h_grid[finite]
    f_grid = f_grid[finite]
    if h_grid.size < 2 or not np.isfinite(h_eval):
        return np.nan

    order = np.argsort(h_grid)
    h_grid = h_grid[order]
    f_grid = f_grid[order]
    if h_eval < h_grid[0] or h_eval > h_grid[-1]:
        return np.nan

    df_dh = np.gradient(f_grid, h_grid)
    return float(np.interp(float(h_eval), h_grid, df_dh))


def thermal_sharpness_rows(results, *, split="global", hc=1.0):
    rows_by_kind = {}
    for kind in ("ipr", "hs"):
        rows = []
        for meta in results["by_kind"][kind]["gram_metadata"]:
            decision = meta["decision_function"]
            h = np.asarray(decision["x"], dtype=float)
            f = np.asarray(decision[f"{split}_mean"], dtype=float)
            h_star = find_decision_boundary(h, f, h_c=hc)
            rows.append({
                "beta": float(meta["beta"]),
                "sharpness": abs(decision_derivative_at(h, f, h_star)),
            })
        rows_by_kind[kind] = sorted(rows, key=lambda row: row["beta"], reverse=True)
    return rows_by_kind


def plot_thermal_sharpness_by_beta(results_by_L):
    set_ipr_plot_style()
    L_values = [20, 50, 100]
    colors = {"ipr": "#1b9e77", "hs": "#d95f02"}
    labels = {"ipr": "IPR global", "hs": "HS global"}
    markers = {"ipr": "s", "hs": "o"}

    fig, axes = plt.subplots(1, len(L_values), figsize=(11.0, 3.4), sharex=True, sharey=True, constrained_layout=True)
    for ax, L in zip(axes, L_values):
        rows_by_kind = thermal_sharpness_rows(results_by_L[L], split="global", hc=1.0)
        for kind in ("ipr", "hs"):
            rows = rows_by_kind[kind]
            betas = np.asarray([row["beta"] for row in rows], dtype=float)
            sharpness = np.asarray([row["sharpness"] for row in rows], dtype=float)
            ax.semilogy(
                betas,
                sharpness,
                marker=markers[kind],
                ms=4.0,
                lw=1.8,
                color=colors[kind],
                label=labels[kind],
            )

        ax.set_xlim(2.0, 0.0)
        ax.set_xlabel(r"$\beta$")
        ax.text(0.05, 0.08, rf"$L={L}$", transform=ax.transAxes, ha="left", va="bottom", fontsize=13)
        ax.tick_params(direction="in", which="both", top=True, right=True)
        ax.grid(which="major", alpha=0.18, lw=0.7)
        ax.grid(which="minor", alpha=0.07, lw=0.5)
        ax.minorticks_on()

    axes[0].set_ylabel(r"$S(\beta)=|\partial_h f|_{h=h^*}$")
    axes[-1].legend(frameon=False, loc="lower right")
    fig.suptitle(r"Sharpness at decision crossing $h^*$", y=1.04)
    return fig, axes


fig, axes = plot_thermal_sharpness_by_beta(thermal_svm_results_by_L)
save_png(fig, "thermal_global_sharpness_at_crossing_L20_L50_L100")